# Day 51: How LLM Apps Work

**Today's question:** what does a large language model (LLM) actually *do*, and what does a program that uses one
look like?

No API key today: we build everything with small pretend models in plain Python.
Before running a cell marked 🤔, predict what it will show.

In [ ]:
import csv
import random
from collections import Counter
from pathlib import Path

## 1. Next-token prediction: a very well-read autocomplete

Your phone suggests the next word. An LLM does the same job, extremely well: it has read a huge amount of text and
predicts **the next piece of text**, over and over.

Let's build a *toy* version that learns "which word follows which" from six sentences.

In [ ]:
corpus = [
    "the battery lasts all day",
    "the battery is great",
    "the battery is weak",
    "the sound is great",
    "the sound is clear",
    "the case is cheap",
]

next_words = {}  # word -> list of the words that came after it
for sentence in corpus:
    words = sentence.split()
    for i in range(len(words) - 1):
        if words[i] not in next_words:
            next_words[words[i]] = []
        next_words[words[i]].append(words[i + 1])

print(next_words["the"])
print(next_words["is"])

This is the model's "training": counting what it has seen. Prediction = pick the most common follower.

In [ ]:
def predict_next(word):
    """Return the word most often seen after `word`."""
    counts = Counter(next_words[word])
    return counts.most_common(1)[0][0]   # most_common(1) -> [("battery", 3)]


print(predict_next("the"))

🤔 **Predict:** what comes after `"is"`?

In [ ]:
print(predict_next("is"))

Generating text = predict a word, add it, predict again from the new last word, until there is nothing to predict.

In [ ]:
def generate(start, max_words=8):
    """Build a sentence by repeatedly predicting the next word."""
    words = [start]
    while words[-1] in next_words and len(words) < max_words:
        words.append(predict_next(words[-1]))
    return " ".join(words)


print(generate("the"))
print(generate("sound"))

Our toy looks at **one** previous word. A real LLM looks at **all the text so far** and learned from a gigantic
amount of text, but its job is the same: *predict what comes next*. It does not look facts up.

## 2. Temperature: how adventurous is the choice?

`next_words["the"]` contains "battery" 3 times out of 6, so `random.choice` picks it about half of the time.
Picking randomly from the list = picking in proportion to how likely each word is.

In [ ]:
random.seed(51)  # fixed seed so this demo shows the same "random" choices every time
picks = [random.choice(next_words["the"]) for i in range(6)]
print(picks)

In [ ]:
def generate_with_temperature(start, temperature, max_words=8):
    """temperature 0: always the most likely word. Higher: pick randomly by likelihood."""
    words = [start]
    while words[-1] in next_words and len(words) < max_words:
        if temperature == 0:
            words.append(predict_next(words[-1]))
        else:
            words.append(random.choice(next_words[words[-1]]))
    return " ".join(words)


random.seed(7)
for i in range(3):
    print("temperature 0:", generate_with_temperature("the", 0))
for i in range(3):
    print("temperature 1:", generate_with_temperature("the", 1))

- **Temperature 0:** same input → same output. Predictable.
- **Higher temperature:** more variety, more surprises, more mistakes.
- Chat assistants sample, so the same question can get different answers. **One run is not a test.**

## 3. Tokens

Models read and write **tokens**: short words, pieces of longer words, punctuation. For example (illustrative,
every model splits differently):

| Text | Possible tokens |
|------|-----------------|
| `The sound is great` | `The` · ` sound` · ` is` · ` great` |
| `unbelievably` | `un` · `belie` · `vably` |
| `Pairing took 10 seconds.` | `P` · `airing` · ` took` · ` 10` · ` seconds` · `.` |

Rule of thumb for English: **1 token ≈ 4 characters ≈ ¾ of a word**. Tokens matter because they are how models
are **limited** (context window) and **billed** (Day 53).

In [ ]:
def estimate_tokens(text):
    """Rough token estimate for English text: about 4 characters per token."""
    return round(len(text) / 4)


print(estimate_tokens("The sound is great"))

In [ ]:
reviews_path = Path("../../../datasets/reviews.csv")
with open(reviews_path, encoding="utf-8", newline="") as file:
    reviews = list(csv.DictReader(file))

all_text = " ".join([row["review_text"] for row in reviews])
print(len(reviews), "reviews,", len(all_text), "characters, about", estimate_tokens(all_text), "tokens")

## 4. Messages and the context window

A request to a chat model is data we already know: a **system** text and a **list of message dicts**.

In [ ]:
system = "You are a friendly assistant for an online shop. Answer in one short sentence."
messages = [
    {"role": "user", "content": "Is the backpack waterproof?"},
]
for message in messages:
    print(message["role"], "->", message["content"])

- **system:** the app's standing instructions, written by the developer.
- **user:** what the person typed. **assistant:** what the model answered before.

The model has **no memory between requests**. A conversation works because the app re-sends the whole history:

In [ ]:
messages = [
    {"role": "user", "content": "Is the backpack waterproof?"},
    {"role": "assistant", "content": "One reviewer says everything stayed dry during a rainy week."},
    {"role": "user", "content": "And does a laptop fit?"},
]
history_text = system + " ".join([message["content"] for message in messages])
print("This request is about", estimate_tokens(history_text), "tokens")

🤔 **Predict:** without the first two messages, could the model know what "And does a laptop fit?" is about?

The **context window** is the most tokens one request may contain (input *and* reply). The course models allow
about 1 million tokens (Oct 2026), but every token costs money, so apps keep prompts short.

## 5. Limitations: confident is not the same as correct

A model always produces *likely-looking* text, even when it has no idea. This pretend model shows the problem:

In [ ]:
def overconfident_llm(prompt):
    """A pretend model that never admits it does not know."""
    if "warranty" in prompt.lower():
        return "The warranty is 5 years."
    return "Yes, definitely."


print(overconfident_llm("How long is the warranty on the desk lamp?"))

Nothing in our data mentions a warranty. This is a **hallucination**: fluent, confident, invented.

| Limitation | So our app must… |
|------------|------------------|
| Hallucination | not present model "facts" as truth without a source or a check |
| Knowledge cut-off (only knows its training data) | put recent or private information *in the prompt* |
| No guarantee of understanding (sarcasm, maths) | test on many inputs and validate output |
| Randomness | check the output format every single time |
| Bias, privacy | review outputs; never send personal data it does not need |

**The model is one unreliable part. The code around it makes the app reliable.**

## 6. The LLM app pipeline

```
input (review) → prompt template → model → parse → output
```

**Step 1: the prompt template.** A function that fills an f-string with today's input.

In [ ]:
def build_prompt(review):
    """Fill the sentiment prompt template with one review."""
    return f"""Classify the sentiment of this product review.
Answer in the form "Sentiment: Positive", "Sentiment: Negative" or "Sentiment: Neutral".

Review: {review}"""


print(build_prompt("Great sound and the battery lasts all day."))

**Step 2: the model.** Any function *text → text* can play the model. Ours counts a few keywords.
Tomorrow we swap in a better mock, and on Day 53 the real thing, without changing the other steps.

In [ ]:
POSITIVE = ["great", "love", "comfortable", "perfect", "easy", "recommended", "best", "good"]
NEGATIVE = ["broke", "leaks", "cracked", "terrible", "flickers", "disappointed", "poor", "stopped"]


def fake_llm(prompt):
    """A pretend LLM: counts positive and negative words in the prompt."""
    cleaned = prompt.lower().replace(".", " ").replace(",", " ").replace("!", " ")
    score = 0
    for word in cleaned.split():
        if word in POSITIVE:
            score += 1
        elif word in NEGATIVE:
            score -= 1
    if score > 0:
        return "Sentiment: Positive"
    if score < 0:
        return "Sentiment: Negative"
    return "Sentiment: Neutral"


reply = fake_llm(build_prompt("Great sound and the battery lasts all day."))
print(reply)

**Step 3: parse.** The rest of the program wants `"positive"`, not a sentence. Never trust the format blindly.

In [ ]:
ALLOWED = ["positive", "negative", "neutral"]


def parse_sentiment(reply):
    """Turn 'Sentiment: Positive' into 'positive'. Unexpected replies become 'unknown'."""
    if ":" not in reply:
        return "unknown"
    label = reply.split(":")[1].strip().lower()
    if label not in ALLOWED:
        return "unknown"
    return label


print(parse_sentiment("Sentiment: Positive"))
print(parse_sentiment("I think the customer is quite happy!"))

🤔 **Predict:** if the template started with *"You are a great assistant."*, what would happen to every result?

In [ ]:
def build_prompt_great(review):
    return f"You are a great assistant.\n{build_prompt(review)}"


neutral_review = "Does the job. Nothing special."
print(parse_sentiment(fake_llm(build_prompt(neutral_review))))
print(parse_sentiment(fake_llm(build_prompt_great(neutral_review))))

The model sees **the whole prompt**, instructions included. Tomorrow's *delimiters* help with exactly this.

## 7. Real-world example: the whole pipeline on the reviews

Steps 4–5: run every review through the pipeline, then compare the label with the star rating
(4–5 stars → positive, 1–2 → negative, 3 → neutral).

In [ ]:
def stars_to_label(rating):
    if rating >= 4:
        return "positive"
    if rating <= 2:
        return "negative"
    return "neutral"


agree = 0
for row in reviews:
    label = parse_sentiment(fake_llm(build_prompt(row["review_text"])))
    if label == stars_to_label(int(row["rating"])):
        agree += 1
percent = round(agree / len(reviews) * 100)
print(f"Agrees with the stars on {agree} of {len(reviews)} reviews ({percent}%)")

In [ ]:
for row in reviews[:6]:
    label = parse_sentiment(fake_llm(build_prompt(row["review_text"])))
    print(f"{row['review_id']}  {row['rating']}★  {label:<9} {row['review_text'][:50]}")

A keyword counter is a weak model, but notice that **every other step is already a real LLM app**.
Tomorrow we improve the prompt and measure the improvement.

## 8. Common mistakes (these cells go wrong **on purpose**)

In [ ]:
# Mistake 1: template without the f. No error, but the model gets the wrong text!
def build_prompt_broken(review):
    return "Classify the sentiment of this review: {review}"


print(build_prompt_broken("Great lamp"))

In [ ]:
# Mistake 2: parsing that assumes a colon. Model output is not guaranteed!
reply = "Positive"
label = reply.split(":")[1]

In [ ]:
# Mistake 3: a "model" that prints instead of returning (Day 17)
def printing_llm(prompt):
    print("Sentiment: Positive")


reply = printing_llm("Great lamp")
reply.split(":")

**Fixes:** remember the `f` and print one prompt before sending it · check the format before indexing ·
`return` the reply.

## Recap

1. An LLM predicts the next token again and again; temperature controls how random the choices are.
2. Tokens ≈ 4 characters each. Requests = system message + list of user/assistant messages, within the context window.
3. Models hallucinate, have a knowledge cut-off and no guarantee of understanding, so **apps check the output**.
4. Pipeline: **input → prompt template → model → parse → output**. A model is just a function text → text.

**Next time:** writing better prompts and *measuring* which prompt works best.